# Wildfire Risk Mapping — Day 1: Data, Training, Baseline Eval (Colab GPU)

Run on a **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU).

Steps: pull dataset from Kaggle → verify → fine-tune ResNet18 (timm) → evaluate (accuracy/F1/ROC-AUC/confusion matrix) → download `best_model.pt` + metrics back to the local repo's `models/` and `outputs/` folders.

This notebook mirrors `scripts/dataset.py`, `scripts/model.py`, `scripts/train.py`, `scripts/evaluate.py` so it runs standalone here.

In [12]:
!pip install -q timm grad-cam kaggle scikit-learn

## 1. Kaggle auth + dataset download
Paste your Kaggle access token when prompted (from kaggle.com → Settings → API → Create New Token, or reuse the one saved locally at `~/.kaggle/access_token`). The input is masked and not saved into the notebook output.

In [13]:
# import os
# from getpass import getpass

# access_token = getpass('Paste your Kaggle access token: ')
# os.makedirs('/root/.kaggle', exist_ok=True)
# with open('/root/.kaggle/access_token', 'w') as f:
#     f.write(access_token)
# os.chmod('/root/.kaggle/access_token', 0o600)
# print('Kaggle credentials saved.')

In [14]:
# !kaggle datasets download -d abdelghaniaaba/wildfire-prediction-dataset -p /content/data/raw
# !cd /content/data/raw && unzip -q -o *.zip
# !ls /content/data/raw

In [15]:
import shutil
from pathlib import Path

RAW = Path('/content/data/raw')
DATA = Path('/content/data')

train_dir = next((p for p in RAW.rglob('train') if p.is_dir()), None)
assert train_dir is not None, 'Could not locate train/ folder — inspect /content/data/raw manually'
source_root = train_dir.parent
for split in ('train', 'valid', 'test'):
    src, dst = source_root / split, DATA / split
    if src.is_dir() and not dst.exists():
        shutil.move(str(src), str(dst))
        print(f'moved {src} -> {dst}')
print('done')

done


## 2. Verify class balance + image integrity (Step 1 done-when check)

In [16]:
# from PIL import Image, ImageFile

# # Tolerate near-complete/truncated files instead of rejecting them (matches
# # what the training pipeline does below) — a handful of files in this dataset
# # are truncated by a few bytes.
# ImageFile.LOAD_TRUNCATED_IMAGES = True

# SPLITS = ('train', 'valid', 'test')
# CLASSES = ('wildfire', 'nowildfire')
# ok = True
# for split in SPLITS:
#     for cls in CLASSES:
#         cls_dir = DATA / split / cls
#         files_ = [f for f in cls_dir.iterdir() if f.is_file()]
#         corrupted = []
#         for f in files_:
#             try:
#                 with Image.open(f) as img:
#                     img.convert('RGB').load()
#             except Exception:
#                 corrupted.append(f.name)
#         print(f'{split}/{cls}: {len(files_)} images, {len(corrupted)} corrupted')
#         if corrupted:
#             ok = False
# print('VERIFICATION', 'PASSED' if ok else 'FAILED')

train/wildfire: 15750 images, 0 corrupted
train/nowildfire: 14500 images, 0 corrupted
valid/wildfire: 3480 images, 0 corrupted
valid/nowildfire: 2820 images, 0 corrupted
test/wildfire: 3480 images, 0 corrupted
test/nowildfire: 2820 images, 0 corrupted
VERIFICATION PASSED


## 3. Dataset / DataLoader

In [17]:
import torch
from PIL import ImageFile
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Tolerate near-complete/truncated files (a few exist in this dataset) instead
# of crashing the DataLoader workers.
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

def get_transforms(train):
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

batch_size = 32
train_ds = datasets.ImageFolder(DATA / 'train', transform=get_transforms(True))
valid_ds = datasets.ImageFolder(DATA / 'valid', transform=get_transforms(False))
test_ds = datasets.ImageFolder(DATA / 'test', transform=get_transforms(False))
assert train_ds.class_to_idx == valid_ds.class_to_idx == test_ds.class_to_idx
class_to_idx = train_ds.class_to_idx
print('class_to_idx:', class_to_idx)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

class_to_idx: {'nowildfire': 0, 'wildfire': 1}


## 4. Model: ResNet18 (timm), frozen early layers, fine-tune layer4 + head

In [18]:
import timm
import torch.nn as nn

def build_model(num_classes=2, pretrained=True):
    model = timm.create_model('resnet18', pretrained=pretrained, num_classes=num_classes)
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if name.startswith(('layer4', 'fc')):
            p.requires_grad = True
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
model = build_model().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

device: cuda
trainable params: 8,394,754 / 11,177,538 (75.1%)


## 5. Train with early stopping

In [ ]:
import time, json
from sklearn.metrics import f1_score

os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-4)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds_all.extend(logits.argmax(1).cpu().tolist())
            labels_all.extend(y.cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(labels_all, preds_all)
    acc = sum(p == l for p, l in zip(preds_all, labels_all)) / len(labels_all)
    return avg_loss, acc, f1

epochs, patience = 15, 3
best_f1, no_improve, history = 0.0, 0, []
for epoch in range(1, epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, True)
    val_loss, val_acc, val_f1 = run_epoch(valid_loader, False)
    dt = time.time() - t0
    print(f'epoch {epoch:02d} ({dt:.0f}s) | train loss {tr_loss:.4f} acc {tr_acc:.3f} f1 {tr_f1:.3f} '
          f'| val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f}')
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc, 'train_f1': tr_f1,
                     'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    if val_f1 > best_f1:
        best_f1, no_improve = val_f1, 0
        torch.save({'model_state': model.state_dict(), 'class_to_idx': class_to_idx}, '/content/models/best_model.pt')
        print(f'  -> new best (val f1={val_f1:.3f}), saved')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'early stopping at epoch {epoch}')
            break

json.dump(history, open('/content/outputs/training_history.json', 'w'), indent=2)
print(f'done. best val f1: {best_f1:.3f}')

epoch 01 (150s) | train loss 0.2129 acc 0.923 f1 0.927 | val loss 0.1100 acc 0.959 f1 0.962
  -> new best (val f1=0.962), saved


## 6. Evaluate on held-out test split

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

ckpt = torch.load('/content/models/best_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
wildfire_idx = class_to_idx['wildfire']

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = F.softmax(logits, dim=1)[:, wildfire_idx].cpu()
        preds = logits.argmax(1).cpu()
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(y.tolist())

binary_labels = [1 if l == wildfire_idx else 0 for l in all_labels]
binary_preds = [1 if p == wildfire_idx else 0 for p in all_preds]

acc = accuracy_score(binary_labels, binary_preds)
f1 = f1_score(binary_labels, binary_preds)
auc = roc_auc_score(binary_labels, all_probs)
cm = confusion_matrix(binary_labels, binary_preds).tolist()
report = classification_report(binary_labels, binary_preds, target_names=['nowildfire', 'wildfire'])

print(f'accuracy: {acc:.4f}\nf1: {f1:.4f}\nroc-auc: {auc:.4f}')
print(cm)
print(report)

result = {'accuracy': acc, 'f1': f1, 'roc_auc': auc, 'confusion_matrix': cm,
          'classification_report': report, 'class_to_idx': class_to_idx}
json.dump(result, open('/content/outputs/test_metrics.json', 'w'), indent=2)

## 7. Save results to Google Drive

`google.colab.files.download()` requires Colab's browser frontend, which isn't available when connecting from VS Code to a Colab kernel — so this mounts Drive instead (you'll get a link + auth-code prompt, works over any connection). Files land in `My Drive/wildfire_risk_outputs/`; download them from drive.google.com (or they'll sync automatically if you have Google Drive for Desktop), then place them into this project's `models/` and `outputs/` folders.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/wildfire_risk_outputs'
os.makedirs(dest, exist_ok=True)
shutil.copy('/content/models/best_model.pt', dest)
shutil.copy('/content/outputs/test_metrics.json', dest)
shutil.copy('/content/outputs/training_history.json', dest)
print(f'Copied results to Google Drive: {dest}')